In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Certification-Gated BigQuery Analytics with MCP and Gemini on Agent Platform

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/RiturajSingh2004/arth-semlayer/blob/main/notebooks/certified_bigquery_mcp.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fmcp%2Fcertified_bigquery_mcp.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/RiturajSingh2004/arth-semlayer/main/notebooks/certified_bigquery_mcp.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/RiturajSingh2004/arth-semlayer/blob/main/notebooks/certified_bigquery_mcp.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/RiturajSingh2004/arth-semlayer/blob/main/notebooks/certified_bigquery_mcp.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/RiturajSingh2004/arth-semlayer/blob/main/notebooks/certified_bigquery_mcp.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/RiturajSingh2004/arth-semlayer/blob/main/notebooks/certified_bigquery_mcp.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/RiturajSingh2004/arth-semlayer/blob/main/notebooks/certified_bigquery_mcp.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/RiturajSingh2004/arth-semlayer/blob/main/notebooks/certified_bigquery_mcp.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author(s) |
| --- |
| [Rituraj Singh](https://github.com/RiturajSingh2004) |

## Overview

Giving an LLM raw text-to-SQL access over an analytics warehouse creates silent governance failures: models hallucinate uncertified metric formulas, bypass role-based access control (`RBAC`), leak sensitive columns, and execute unbounded table scans.

This notebook demonstrates how to place a **certification-gated semantic layer (`semlayer`)** in front of BigQuery's managed Model Context Protocol (`MCP`) server (`https://bigquery.googleapis.com/mcp`) and connect it to **Gemini** using the Google Gen AI SDK on Agent Platform (Vertex AI). Instead of exposing raw SQL to the model, a local `stdio` MCP server exposes only certified metrics, foreign-key entity relationships, and governance tools, while compiling parameterized queries through `sqlglot` AST literal binding and enforcing BigQuery `execute_sql_readonly` with a pre-execution `dryRun` byte cap.

In this tutorial, you will:

- Seed the deterministic `jaffle_shop` dataset (`customers`, `orders`, `payments`) into BigQuery (with an optional offline `DuckDB` toggle for zero-billing testing).
- Define a YAML catalog of `certified` vs. `draft` metrics, foreign-key entity edges, and default-deny `RBAC` rules (`row_filter` and `deny_dims`).
- Build a `SimpleCompiler`, foreign-key entity-graph traversal engine (`UNNEST`), `sqlglot` AST literal binder, and `BigQueryMCP` client calling only `execute_sql_readonly` with a `dryRun` byte cap.
- Launch a local `stdio` MCP server exposing seven governed tools (`list_metrics`, `describe_metric`, `query_metric`, `describe_graph`, `traverse_graph`, `report_gap`, `report_issue`) with **no raw SQL tool**, and connect Gemini via `google-genai` tool calling.
- Verify five governed scenarios end-to-end: authorized metric queries, default-deny refusals, row-level filtering, multi-hop entity-graph traversal, and hostile SQL-injection neutralization plus `dryRun` byte-cap enforcement.

## Get started

### Install Google Gen AI SDK and other required packages

In [ ]:
%pip install --upgrade --quiet google-genai mcp sqlglot pydantic pyyaml duckdb

### Authenticate your notebook environment

If you are running this notebook in **Google Colab**, run the cell below to authenticate your account.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

To get started using Agent Platform (Vertex AI), you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com) as well as the [BigQuery API](https://console.cloud.google.com/flows/enableapi?apiid=bigquery.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
import os

from google import genai

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "us-central1"  # @param {type: "string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT", ""))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", LOCATION)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID or "[your-project-id]",
    location=LOCATION,
)

### Import libraries

In [ ]:
import datetime as dt
import json
import random
import shutil
import subprocess
import tempfile
from pathlib import Path
from typing import Any

import duckdb
from google.genai import types
from IPython.display import Markdown, display
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

WORK_DIR = Path("semlayer_workspace")
WORK_DIR.mkdir(parents=True, exist_ok=True)

### Load model

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param {type: "string"}

## 1. Seed the deterministic `jaffle_shop` dataset

We first generate the deterministic `jaffle_shop` tables (`customers`, `orders`, `payments`) locally in `DuckDB` using a fixed seed (`42`), then load them into BigQuery when `WAREHOUSE_BACKEND = "bigquery"`.

- **BigQuery mode (`"bigquery"`)**: Loads newline-delimited JSON directly into `{PROJECT_ID}.{BQ_DATASET}` via BigQuery load jobs (works on the BigQuery free tier without requiring DML or billing).
- **Offline mode (`"duckdb"`)**: Uses the local read-only `DuckDB` database so you can run and test every semantic layer guardrail without Google Cloud billing or credentials.

In [ ]:
WAREHOUSE_BACKEND = "bigquery"  # @param ["bigquery", "duckdb"]
BQ_DATASET = "jaffle_shop"  # @param {type: "string"}
BQ_LOCATION = "US"  # @param {type: "string"}
BQ_MAX_BYTES = 100000000  # @param {type: "integer"}

DUCKDB_PATH = WORK_DIR / "jaffle.duckdb"

BQ_SCHEMAS: dict[str, str] = {
    "customers": "id:INT64,name:STRING,email:STRING,region:STRING,created_at:DATE",
    "orders": "id:INT64,customer_id:INT64,status:STRING,order_date:DATE",
    "payments": ",".join(
        [
            "id:INT64",
            "order_id:INT64",
            "customer_id:INT64",
            "method:STRING",
            "status:STRING",
            "amount:FLOAT64",
            "paid_at:DATE",
        ]
    ),
}


def build_jaffle_duckdb(db_path: Path) -> None:
    """Create deterministic jaffle_shop tables (customers, orders, payments)."""
    db_path.unlink(missing_ok=True)
    rng = random.Random(42)  # noqa: S311
    today = dt.datetime.now(dt.timezone.utc).date()

    connection = duckdb.connect(str(db_path))
    try:
        connection.execute(
            "CREATE TABLE customers("
            "id INTEGER, name VARCHAR, email VARCHAR, region VARCHAR, created_at DATE)"
        )
        connection.execute(
            "CREATE TABLE orders("
            "id INTEGER, customer_id INTEGER, status VARCHAR, order_date DATE)"
        )
        connection.execute(
            "CREATE TABLE payments("
            "id INTEGER, order_id INTEGER, customer_id INTEGER, method VARCHAR, "
            "status VARCHAR, amount DOUBLE, paid_at DATE)"
        )

        customers = [
            (
                idx,
                f"Customer {idx}",
                f"c{idx}@example.com",
                rng.choice(["EU", "US", "APAC"]),
                today - dt.timedelta(days=rng.randint(0, 365)),
            )
            for idx in range(1, 61)
        ]
        orders = []
        payments = []
        for idx in range(1, 301):
            customer_id = rng.randint(1, 60)
            order_day = today - dt.timedelta(days=rng.randint(0, 90))
            order_status = rng.choices(
                ["completed", "returned", "placed", "shipped"],
                weights=[60, 10, 15, 15],
            )[0]
            orders.append((idx, customer_id, order_status, order_day))

            payment_method = rng.choice(["card", "bank", "gift"])
            payment_status = rng.choices(["success", "failed"], weights=[85, 15])[0]
            payment_amount = round(rng.uniform(5, 200), 2)
            payments.append(
                (
                    idx,
                    idx,
                    customer_id,
                    payment_method,
                    payment_status,
                    payment_amount,
                    order_day,
                )
            )

        connection.executemany("INSERT INTO customers VALUES (?,?,?,?,?)", customers)
        connection.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
        connection.executemany("INSERT INTO payments VALUES (?,?,?,?,?,?,?)", payments)
    finally:
        connection.close()


def seed_bigquery_dataset(
    db_path: Path, project_id: str, dataset_id: str, location: str = "US"
) -> None:
    """Load the deterministic DuckDB tables into BigQuery without DML."""
    connection = duckdb.connect(str(db_path), read_only=True)
    exported_rows: dict[str, list[dict[str, Any]]] = {}
    try:
        for table_name in BQ_SCHEMAS:
            cursor = connection.execute(f"SELECT * FROM {table_name}")  # noqa: S608
            col_names = [desc[0] for desc in cursor.description]
            exported_rows[table_name] = [
                {
                    col: (val.isoformat() if hasattr(val, "isoformat") else val)
                    for col, val in zip(col_names, row, strict=True)
                }
                for row in cursor.fetchall()
            ]
    finally:
        connection.close()

    try:
        from google.cloud import bigquery

        bq_client = bigquery.Client(project=project_id, location=location)
        dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
        dataset_ref.location = location
        bq_client.create_dataset(dataset_ref, exists_ok=True)

        for table_name, records in exported_rows.items():
            schema_fields = [
                bigquery.SchemaField(
                    part.split(":")[0], part.split(":")[1], mode="NULLABLE"
                )
                for part in BQ_SCHEMAS[table_name].split(",")
            ]
            job_config = bigquery.LoadJobConfig(
                schema=schema_fields,
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
            )
            job = bq_client.load_table_from_json(
                records, f"{project_id}.{dataset_id}.{table_name}", job_config=job_config
            )
            job.result()
        return
    except ImportError:
        pass

    bq_bin = shutil.which("bq") or shutil.which("bq.cmd")
    if not bq_bin:
        raise RuntimeError(
            "Install `google-cloud-bigquery` or Google Cloud SDK (`bq`) to seed BigQuery."
        )
    subprocess.run(  # noqa: S603
        [
            bq_bin,
            f"--location={location}",
            f"--project_id={project_id}",
            "mk",
            "--dataset",
            "--force",
            f"{project_id}:{dataset_id}",
        ],
        check=True,
    )
    with tempfile.TemporaryDirectory() as tmp_dir:
        for table_name, records in exported_rows.items():
            jsonl_path = Path(tmp_dir) / f"{table_name}.jsonl"
            jsonl_path.write_text(
                "\n".join(json.dumps(rec) for rec in records) + "\n",
                encoding="utf-8",
            )
            subprocess.run(  # noqa: S603
                [
                    bq_bin,
                    f"--location={location}",
                    f"--project_id={project_id}",
                    "load",
                    "--replace",
                    "--source_format=NEWLINE_DELIMITED_JSON",
                    f"{project_id}:{dataset_id}.{table_name}",
                    str(jsonl_path),
                    BQ_SCHEMAS[table_name],
                ],
                check=True,
            )


build_jaffle_duckdb(DUCKDB_PATH)
print(f"Seeded local DuckDB replica at: {DUCKDB_PATH}")

if WAREHOUSE_BACKEND == "bigquery":
    if not PROJECT_ID or PROJECT_ID in {"[your-project-id]", "None"}:
        print(
            "PROJECT_ID is not configured; falling back to WAREHOUSE_BACKEND='duckdb' "
            "for local execution. Set PROJECT_ID above to run against BigQuery MCP."
        )
        ACTIVE_BACKEND = "duckdb"
    else:
        seed_bigquery_dataset(DUCKDB_PATH, PROJECT_ID, BQ_DATASET, BQ_LOCATION)
        ACTIVE_BACKEND = "bigquery"
        print(f"Seeded BigQuery dataset: {PROJECT_ID}.{BQ_DATASET}")
else:
    ACTIVE_BACKEND = "duckdb"
    print("Using offline DuckDB backend.")

## 2. Define the certification-gated semantic layer

Unlike raw text-to-SQL prompts, `semlayer` separates **authored definitions** from **certified runtime execution**:

1. **Certification gate**: Only definitions in `state: certified` with a designated `owner` are served. Draft or deprecated definitions (such as `revenue_unfiltered`) are invisible to agents and refused at query time.
2. **Default-deny `RBAC`**: Roles receive zero access unless an explicit `certified` `allow` rule matches. `deny` beats `allow`, `row_filter` predicates (validated via `sqlglot` so subqueries and commands cannot be smuggled in) restrict rows automatically, and `deny_dims` blocks sensitive dimensions like `email`.
3. **Typed SQL compilation (`SimpleCompiler` + `sqlglot`)**: Metric expressions, dimensions, foreign-key `UNNEST` filters, and user filter values are bound as `sqlglot` AST literal nodes and verified to emit a single `SELECT` statement before reaching BigQuery.

In [ ]:
CATALOG_PATH = WORK_DIR / "catalog.yaml"

CATALOG_YAML = """
metrics:
  - name: revenue
    description: Total successful payment amount (USD).
    state: certified
    owner: alice
    version: 2
    table: payments
    agg: sum
    column: amount
    filters:
      - "status = 'success'"
    dimensions:
      - method
    entity: Customer
    entity_key: customer_id
    lineage:
      - payments.amount
      - payments.status

  - name: revenue_unfiltered
    description: Uncertified sum of all payment rows including failed payments.
    state: draft
    owner: bob
    version: 0
    table: payments
    agg: sum
    column: amount
    filters: []
    dimensions: []

  - name: customer_count
    description: Total number of customer records in the customers table.
    state: certified
    owner: carol
    version: 1
    table: customers
    agg: count
    column: "*"
    filters: []
    dimensions:
      - region

  - name: order_count
    description: Total number of orders placed across all order statuses.
    state: certified
    owner: bob
    version: 1
    table: orders
    agg: count
    column: "*"
    filters: []
    dimensions:
      - status
    entity: Order
    entity_key: id

  - name: avg_payment
    description: Average monetary amount of successful payment transactions.
    state: certified
    owner: carol
    version: 1
    table: payments
    agg: avg
    column: amount
    filters:
      - "status = 'success'"
    dimensions: []

  - name: active_customers_30d
    description: Unique customers who placed at least one order in the last 30 days.
    state: certified
    owner: alice
    version: 1
    table: orders
    agg: count_distinct
    column: customer_id
    filters:
      - "order_date >= CURRENT_DATE - 30"
    dimensions: []

entities:
  - name: Customer
    state: certified
    owner: data-team
    version: 1
    table: customers
    key: id
  - name: Order
    state: certified
    owner: data-team
    version: 1
    table: orders
    key: id
  - name: Payment
    state: certified
    owner: data-team
    version: 1
    table: payments
    key: id

edges:
  - name: customer_has_order
    state: certified
    owner: data-team
    version: 1
    src: Customer
    rel: HAS_ORDER
    dst: Order
    src_key: id
    dst_key: customer_id
  - name: customer_has_payment
    state: certified
    owner: data-team
    version: 1
    src: Customer
    rel: HAS_PAYMENT
    dst: Payment
    src_key: id
    dst_key: customer_id
  - name: order_has_payment
    state: certified
    owner: data-team
    version: 1
    src: Order
    rel: HAS_PAYMENT
    dst: Payment
    src_key: id
    dst_key: order_id

rules:
  - name: finance_all
    state: certified
    owner: security-team
    version: 1
    role: finance
    metric: "*"
    effect: allow
  - name: finance_restrict_sensitive
    state: certified
    owner: security-team
    version: 1
    role: finance
    metric: "*"
    effect: deny
    deny_dims:
      - email
  - name: eu_analyst_customers
    state: certified
    owner: security-team
    version: 1
    role: eu_analyst
    metric: customer_count
    effect: allow
    row_filter: "region = 'EU'"
  - name: eu_analyst_orders
    state: certified
    owner: security-team
    version: 1
    role: eu_analyst
    metric: order_count
    effect: allow
  - name: eu_analyst_restrict_sensitive
    state: certified
    owner: security-team
    version: 1
    role: eu_analyst
    metric: "*"
    effect: deny
    deny_dims:
      - email
"""

CATALOG_PATH.write_text(CATALOG_YAML.strip() + "\n", encoding="utf-8")
print(f"Wrote certified catalog to: {CATALOG_PATH}")

### Implement the semantic compiler, `sqlglot` AST literal binder, and `BigQueryMCP` client

Next, we write a self-contained module (`semlayer_workspace/semlayer_engine.py`) so both the interactive notebook cells and the local `stdio` MCP server subprocess share the exact same governance engine:

- `evaluate_policy`: Enforces default-deny `RBAC`, `row_filter` composition, and `deny_dims` restrictions.
- `SimpleCompiler` and `EntityGraph`: Compiles certified metrics and multi-hop foreign-key traversals (`IN UNNEST(?)` for BigQuery GoogleSQL, `list_contains(?, ...)` for `DuckDB`).
- `render_bigquery_sql`: Parses DuckDB-dialect SQL via `sqlglot`, binds every `?` parameter as a typed `sqlglot.exp` AST literal node, rewrites date offsets, qualifies `` `project`.`dataset`.`table` ``, and verifies the rendered GoogleSQL is a single `SELECT` query with zero unbound placeholders.
- `BigQueryMCP`: Connects over streamable HTTP to `https://bigquery.googleapis.com/mcp`, ensures `execute_sql_readonly` is exposed, enforces a `dryRun` byte cap (`max_bytes`) via `execute_sql_readonly(..., dryRun=True)` (falling back to the BigQuery `queries` REST `dryRun` endpoint if needed), and rejects incomplete or truncated responses.

In [ ]:
%%writefile semlayer_workspace/semlayer_engine.py
"""Self-contained certification-gated semantic layer and BigQuery MCP client."""

from __future__ import annotations

import asyncio
import concurrent.futures
import datetime as dt
import decimal
import json
import math
import re
import urllib.error
import urllib.request
from collections import defaultdict
from dataclasses import dataclass, field
from enum import Enum
from typing import TYPE_CHECKING, Any, Literal

import duckdb
import sqlglot
import yaml
from mcp import ClientSession
from mcp.client.streamable_http import create_mcp_http_client, streamable_http_client
from pydantic import BaseModel, Field, field_validator
from sqlglot import exp

if TYPE_CHECKING:
    from collections.abc import Callable, Sequence
    from pathlib import Path

IDENT_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
CLOUD_SCOPE = "https://www.googleapis.com/auth/cloud-platform"
BQ_MCP_URL = "https://bigquery.googleapis.com/mcp"
ALLOWED_BQ_TOOL = "execute_sql_readonly"
MAX_RESULT_ROWS = 1000


def ident(value: str) -> str:
    """Validate that a string is a safe SQL identifier."""
    if not IDENT_RE.match(value):
        raise ValueError(f"invalid SQL identifier: {value!r}")
    return value


class State(str, Enum):
    """Lifecycle states for semantic definitions."""

    DRAFT = "draft"
    CERTIFIED = "certified"
    DEPRECATED = "deprecated"


class Metric(BaseModel):
    """Governed metric definition."""

    name: str
    description: str = ""
    state: State = State.DRAFT
    owner: str | None = None
    version: int = 0
    table: str
    agg: Literal["sum", "count", "count_distinct", "avg", "min", "max"]
    column: str = "*"
    filters: list[str] = Field(default_factory=list)
    dimensions: list[str] = Field(default_factory=list)
    entity: str | None = None
    entity_key: str | None = None
    lineage: list[str] = Field(default_factory=list)

    @field_validator("name", "table")
    @classmethod
    def _validate_ident(cls, value: str) -> str:
        return ident(value)

    @field_validator("column")
    @classmethod
    def _validate_column(cls, value: str) -> str:
        return value if value == "*" else ident(value)

    @field_validator("dimensions")
    @classmethod
    def _validate_dims(cls, values: list[str]) -> list[str]:
        return [ident(item) for item in values]

    @property
    def expression(self) -> str:
        """Return the SQL aggregate expression for this metric."""
        if self.agg == "count_distinct":
            return f"COUNT(DISTINCT {self.column})"
        if self.agg == "count":
            return f"COUNT({self.column})"
        return f"{self.agg.upper()}({self.column})"


class Entity(BaseModel):
    """Business entity mapped to a primary table key."""

    name: str
    state: State = State.DRAFT
    owner: str | None = None
    version: int = 0
    table: str
    key: str


class Edge(BaseModel):
    """Directed foreign-key relationship between two certified entities."""

    name: str
    state: State = State.DRAFT
    owner: str | None = None
    version: int = 0
    src: str
    rel: str
    dst: str
    src_key: str
    dst_key: str


class Rule(BaseModel):
    """Role-based access rule (default-deny; certified rules only; deny beats allow)."""

    name: str
    state: State = State.DRAFT
    owner: str | None = None
    version: int = 0
    role: str
    metric: str = "*"
    effect: Literal["allow", "deny"] = "allow"
    row_filter: str | None = None
    deny_dims: list[str] = Field(default_factory=list)

    @field_validator("row_filter")
    @classmethod
    def _validate_row_filter(cls, value: str | None) -> str | None:
        if value is None:
            return None
        try:
            tree = sqlglot.condition(value)
        except sqlglot.errors.SqlglotError as exc:
            raise ValueError(f"invalid row_filter: {exc}") from exc
        if tree.find(exp.Select, exp.Subquery, exp.Command):
            raise ValueError("row_filter must be a plain scalar predicate")
        return value


@dataclass
class Catalog:
    """In-memory registry loaded from YAML."""

    metrics: dict[str, Metric]
    entities: dict[str, Entity]
    edges: list[Edge]
    rules: list[Rule]

    @classmethod
    def from_yaml(cls, path: Path) -> Catalog:
        """Load and validate catalog definitions from a YAML file."""
        raw = yaml.safe_load(path.read_text(encoding="utf-8"))
        metrics = {item["name"]: Metric(**item) for item in raw.get("metrics", [])}
        entities = {item["name"]: Entity(**item) for item in raw.get("entities", [])}
        edges = [Edge(**item) for item in raw.get("edges", [])]
        rules = [Rule(**item) for item in raw.get("rules", [])]
        return cls(metrics=metrics, entities=entities, edges=edges, rules=rules)

    def certified_metrics(self) -> list[Metric]:
        """Return all certified metrics that have a designated owner."""
        return [
            metric
            for metric in self.metrics.values()
            if metric.state is State.CERTIFIED and metric.owner
        ]

    def certified_entities(self) -> list[Entity]:
        """Return all certified entities."""
        return [
            entity
            for entity in self.entities.values()
            if entity.state is State.CERTIFIED
        ]

    def certified_edges(self) -> list[Edge]:
        """Return all certified foreign-key edges."""
        return [edge for edge in self.edges if edge.state is State.CERTIFIED]

    def certified_rules(self) -> list[Rule]:
        """Return all certified access-control rules."""
        return [rule for rule in self.rules if rule.state is State.CERTIFIED]


@dataclass
class PolicyDecision:
    """Result of evaluating RBAC rules for a (role, metric) pair."""

    allowed: bool
    reason: str = ""
    row_filter: str | None = None
    deny_dims: set[str] = field(default_factory=set)


def evaluate_policy(rules: list[Rule], role: str, metric: str) -> PolicyDecision:
    """Evaluate default-deny RBAC using certified rules only."""
    matching = [
        rule
        for rule in rules
        if rule.state is State.CERTIFIED
        and rule.role == role
        and rule.metric in ("*", metric)
    ]
    denied_dims = {dim for rule in matching for dim in rule.deny_dims}
    if any(rule.effect == "deny" and not rule.deny_dims for rule in matching):
        return PolicyDecision(False, "denied by explicit rule", deny_dims=denied_dims)
    allows = [rule for rule in matching if rule.effect == "allow"]
    if not allows:
        return PolicyDecision(
            False, f"no allow rule for role '{role}'", deny_dims=denied_dims
        )
    if any(rule.row_filter is None for rule in allows):
        return PolicyDecision(True, row_filter=None, deny_dims=denied_dims)
    combined_filter = " OR ".join(
        sorted({f"({rule.row_filter})" for rule in allows if rule.row_filter})
    )
    return PolicyDecision(True, row_filter=combined_filter, deny_dims=denied_dims)


def has_unrestricted_access(rules: list[Rule], role: str) -> bool:
    """Check whether a role holds unrestricted wildcard access for entity-key traversal."""
    active = [
        rule for rule in rules if rule.state is State.CERTIFIED and rule.role == role
    ]
    if any(rule.effect == "deny" and not rule.deny_dims for rule in active):
        return False
    return any(
        rule.effect == "allow" and rule.metric == "*" and rule.row_filter is None
        for rule in active
    )


class CompileError(ValueError):
    """Raised when a metric request violates certified dimensions or entity bindings."""


@dataclass
class CompiledQuery:
    """Parameterized SQL query produced by SimpleCompiler."""

    sql: str
    params: list[Any] = field(default_factory=list)


class SimpleCompiler:
    """Compile a certified Metric into parameterized SQL."""

    def __init__(self, dialect: str = "duckdb") -> None:
        self.dialect = dialect

    def compile(
        self,
        metric: Metric,
        dims: Sequence[str] | None = None,
        filters: dict[str, Any] | None = None,
        row_filter: str | None = None,
        entity_keys: Sequence[Any] | None = None,
    ) -> CompiledQuery:
        """Compile metric SQL with validated dimensions, filters, row_filter, and entity_keys."""
        dim_list = list(dims or [])
        filter_map = dict(filters or {})
        disallowed = [
            col for col in [*dim_list, *filter_map] if col not in metric.dimensions
        ]
        if disallowed:
            msg = (
                f"not allowed for {metric.name}: {disallowed}; "
                f"allowed: {metric.dimensions}"
            )
            raise CompileError(msg)
        dim_list = [col for col in dim_list if col not in filter_map]
        where_clauses = [f"({item})" for item in metric.filters]
        params: list[Any] = []

        for col, val in filter_map.items():
            where_clauses.append(f"{ident(col)} = ?")
            params.append(val)

        if row_filter:
            where_clauses.append(f"({row_filter})")

        if entity_keys is not None:
            if not metric.entity_key:
                raise CompileError(f"{metric.name} is not linked to an entity")
            key_list = list(entity_keys)
            if self.dialect == "bigquery":
                if not key_list:
                    where_clauses.append("FALSE")
                else:
                    where_clauses.append(f"{ident(metric.entity_key)} IN UNNEST(?)")
                    params.append(key_list)
            else:
                where_clauses.append(f"list_contains(?, {ident(metric.entity_key)})")
                params.append(key_list)

        select_cols = ", ".join(
            [*map(ident, dim_list), f"{metric.expression} AS {ident(metric.name)}"]
        )
        sql = f"SELECT {select_cols} FROM {ident(metric.table)}"  # noqa: S608
        if where_clauses:
            sql += " WHERE " + " AND ".join(where_clauses)
        if dim_list:
            group_cols = ", ".join(map(ident, dim_list))
            sql += f" GROUP BY {group_cols} ORDER BY {group_cols}"
        return CompiledQuery(sql=sql, params=params)


class EntityGraph:
    """Traverse certified foreign-key relationships across entities."""

    def __init__(self, entities: list[Entity], edges: list[Edge]) -> None:
        self.entities = {entity.name: entity for entity in entities}
        self.edges = edges

    def traverse_sql(
        self,
        start: str,
        path: list[str],
        filters: dict[str, dict[str, Any]] | None = None,
    ) -> CompiledQuery:
        """Build a parameterized SELECT DISTINCT query over a foreign-key path."""
        if start not in self.entities:
            raise ValueError(f"unknown entity {start!r}")
        chain = [self.entities[start]]
        current = start
        joins: list[str] = []

        for rel_name in path:
            edge = next(
                (
                    candidate
                    for candidate in self.edges
                    if candidate.src == current and candidate.rel == rel_name
                ),
                None,
            )
            if edge is None:
                raise ValueError(f"no relationship {rel_name!r} from {current!r}")
            next_entity = self.entities[edge.dst]
            hop = len(chain)
            join_clause = (
                f"JOIN {ident(next_entity.table)} e{hop} "
                f"ON e{hop - 1}.{ident(edge.src_key)} = e{hop}.{ident(edge.dst_key)}"
            )
            joins.append(join_clause)
            chain.append(next_entity)
            current = edge.dst

        where_clauses: list[str] = []
        params: list[Any] = []
        for entity_name, conditions in (filters or {}).items():
            indices = [
                idx for idx, item in enumerate(chain) if item.name == entity_name
            ]
            if not indices:
                raise ValueError(f"{entity_name!r} is not on the traversal path")
            for col, val in conditions.items():
                where_clauses.append(f"e{indices[0]}.{ident(col)} = ?")
                params.append(val)

        root = chain[0]
        base_select = (
            f"SELECT DISTINCT e0.{ident(root.key)} FROM {ident(root.table)} e0"  # noqa: S608
        )
        sql = f"{base_select} {' '.join(joins)}".strip()
        if where_clauses:
            sql += " WHERE " + " AND ".join(where_clauses)
        sql += " ORDER BY 1 LIMIT 10000"
        return CompiledQuery(sql=sql, params=params)


def _scalar_literal(value: Any) -> exp.Expression:
    if value is None:
        return exp.Null()
    if isinstance(value, bool):
        return exp.Boolean(this=value)
    if isinstance(value, int):
        return (
            exp.Neg(this=exp.Literal.number(-value))
            if value < 0
            else exp.Literal.number(value)
        )
    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(f"non-finite float literal not allowed: {value!r}")
        return (
            exp.Neg(this=exp.Literal.number(-value))
            if value < 0
            else exp.Literal.number(value)
        )
    if isinstance(value, dt.date) and not isinstance(value, dt.datetime):
        return exp.Cast(
            this=exp.Literal.string(value.isoformat()),
            to=exp.DataType.build("DATE"),
        )
    if isinstance(value, str):
        if "\x00" in value:
            raise ValueError("null byte is not allowed in BigQuery string literal")
        return exp.Literal.string(value)
    raise ValueError(f"unsupported literal type: {type(value).__name__}")


def _param_literal(value: Any) -> exp.Expression:
    if isinstance(value, (list, tuple)):
        return exp.Array(expressions=[_scalar_literal(item) for item in value])
    return _scalar_literal(value)


def render_bigquery_sql(
    sql: str, params: Sequence[Any] | None, project: str, dataset: str
) -> str:
    """Bind parameters as sqlglot AST literals and emit verified BigQuery GoogleSQL."""
    if "\x00" in sql:
        raise ValueError("null byte is not allowed in SQL")
    ident(dataset)
    if not project or any(ch in project for ch in ("`", ";", "\n", "\r", "\x00", " ")):
        raise ValueError(f"invalid BigQuery project identifier: {project!r}")

    statements = sqlglot.parse(sql, read="duckdb")
    if (
        len(statements) != 1
        or not isinstance(statements[0], exp.Select)
        or statements[0].find(exp.Command)
    ):
        raise ValueError("BigQueryMCP allows single SELECT statements only")
    tree = statements[0]

    bound_params = list(params or [])
    placeholders = [
        node for node in tree.walk(bfs=False) if isinstance(node, exp.Placeholder)
    ]
    if len(placeholders) != len(bound_params):
        raise ValueError(
            f"placeholder mismatch: {len(placeholders)} vs {len(bound_params)}"
        )

    for placeholder, val in zip(placeholders, bound_params, strict=True):
        if (
            isinstance(val, (list, tuple))
            and not val
            and isinstance(placeholder.parent, exp.Unnest)
            and isinstance(placeholder.parent.parent, exp.In)
        ):
            placeholder.parent.parent.replace(exp.Boolean(this=False))
            continue
        placeholder.replace(_param_literal(val))

    def _rewrite_dates(node: exp.Expression) -> exp.Expression:
        if (
            isinstance(node, exp.Sub)
            and isinstance(node.this, exp.CurrentDate)
            and isinstance(node.expression, exp.Literal)
            and node.expression.is_int
        ):
            return exp.DateSub(
                this=exp.CurrentDate(),
                expression=node.expression.copy(),
                unit=exp.var("DAY"),
            )
        return node

    tree = tree.transform(_rewrite_dates)

    cte_names = {cte.alias for cte in tree.find_all(exp.CTE) if cte.alias}
    for table_node in tree.find_all(exp.Table):
        if (
            table_node.name
            and table_node.name not in cte_names
            and not table_node.db
            and not table_node.catalog
        ):
            table_node.set("catalog", exp.Identifier(this=project, quoted=True))
            table_node.set("db", exp.Identifier(this=dataset, quoted=True))
            table_node.set("this", exp.Identifier(this=table_node.name, quoted=True))

    rendered = tree.sql(dialect="bigquery")
    verified = sqlglot.parse(rendered, read="bigquery")
    if (
        len(verified) != 1
        or not isinstance(verified[0], exp.Select)
        or list(verified[0].find_all(exp.Placeholder))
    ):
        raise ValueError("rendered GoogleSQL failed post-parse safety check")
    return rendered


@dataclass
class QueryResult:
    """Tabular query output returned by a warehouse backend."""

    columns: list[str]
    rows: list[list[Any]]
    bytes_estimated: int = 0


class DuckDBWarehouse:
    """Local read-only DuckDB warehouse with optional dryRun byte-cap simulation."""

    dialect = "duckdb"

    def __init__(self, db_path: Path, max_bytes: int = 100_000_000) -> None:
        self.db_path = db_path
        self.max_bytes = max_bytes

    def query(
        self,
        sql: str,
        params: Sequence[Any] | None = None,
        max_rows: int = MAX_RESULT_ROWS,
    ) -> QueryResult:
        """Execute parameterized read-only SQL against DuckDB."""
        _ = render_bigquery_sql(sql, params, "demo-project", "jaffle_shop")
        estimated_bytes = max(len(sql.encode("utf-8")) * 32, 2400)
        if self.max_bytes > 0 and estimated_bytes > self.max_bytes:
            raise RuntimeError(
                f"dryRun estimated {estimated_bytes} bytes, above cap {self.max_bytes}"
            )
        connection = duckdb.connect(str(self.db_path), read_only=True)
        try:
            cursor = (
                connection.execute(sql, list(params))
                if params
                else connection.execute(sql)
            )
            columns = [desc[0] for desc in (cursor.description or [])]
            fetched = cursor.fetchmany(max_rows)
            return QueryResult(
                columns=columns,
                rows=[list(row) for row in fetched],
                bytes_estimated=estimated_bytes,
            )
        finally:
            connection.close()


def google_access_token() -> str:
    """Acquire an OAuth2 access token from Application Default Credentials."""
    import google.auth
    import google.auth.transport.requests

    credentials, _ = google.auth.default(scopes=[CLOUD_SCOPE])
    if not credentials.valid or not credentials.token:
        credentials.refresh(google.auth.transport.requests.Request())
    return str(credentials.token)


def _cast_bq_cell(value: Any, bq_type: str) -> Any:
    if value is None:
        return None
    normalized = (bq_type or "STRING").upper()
    if normalized in {"INT64", "INTEGER", "INT", "SMALLINT", "BIGINT"}:
        return int(value)
    if normalized in {"FLOAT64", "FLOAT", "NUMERIC", "BIGNUMERIC", "DECIMAL"}:
        return float(value)
    if normalized in {"BOOL", "BOOLEAN"}:
        return value if isinstance(value, bool) else str(value).lower() == "true"
    return value


def _extract_mcp_payload(result: Any) -> dict[str, Any]:
    is_err = getattr(result, "is_error", False) or getattr(result, "isError", False)
    if is_err:
        text = " ".join(
            getattr(part, "text", str(part)) for part in getattr(result, "content", [])
        )
        raise RuntimeError(f"BigQuery MCP error: {text}")
    structured = getattr(result, "structured_content", None) or getattr(
        result, "structuredContent", None
    )
    if isinstance(structured, dict):
        return structured
    texts = [
        getattr(part, "text", "")
        for part in getattr(result, "content", [])
        if getattr(part, "text", "")
    ]
    if not texts:
        return {"rows": [], "schema": {"fields": []}}
    parsed = json.loads("\n".join(texts).strip())
    return (
        parsed
        if isinstance(parsed, dict)
        else {"rows": parsed, "schema": {"fields": []}}
    )


def _parse_bq_payload(payload: dict[str, Any], estimated_bytes: int = 0) -> QueryResult:
    if errors := (
        payload.get("errors") or payload.get("error") or payload.get("errorResult")
    ):
        raise RuntimeError(f"BigQuery error: {errors}")
    if payload.get("jobComplete") is False:
        raise RuntimeError("BigQuery query job incomplete; refusing partial result")
    if payload.get("pageToken"):
        raise RuntimeError("BigQuery result paginated; refusing truncated result")

    fields = (payload.get("schema") or {}).get("fields") or []
    columns = [field_def["name"] for field_def in fields]
    col_types = [field_def.get("type", "STRING") for field_def in fields]
    raw_rows = payload.get("rows") or []
    total_rows = payload.get("totalRows")
    if total_rows is not None and int(total_rows) > len(raw_rows):
        raise RuntimeError(
            f"BigQuery result truncated: returned {len(raw_rows)} of {total_rows} rows"
        )

    rows: list[list[Any]] = []
    for record in raw_rows:
        if isinstance(record, dict) and isinstance(record.get("f"), list):
            cells = [
                cell.get("v") if isinstance(cell, dict) else cell
                for cell in record["f"]
            ]
            rows.append(
                [
                    _cast_bq_cell(
                        val, col_types[idx] if idx < len(col_types) else "STRING"
                    )
                    for idx, val in enumerate(cells)
                ]
            )
        elif isinstance(record, dict):
            if not columns:
                columns = list(record.keys())
                col_types = ["STRING"] * len(columns)
            rows.append(
                [
                    _cast_bq_cell(record.get(col), col_types[idx])
                    for idx, col in enumerate(columns)
                ]
            )
    return QueryResult(columns=columns, rows=rows, bytes_estimated=estimated_bytes)


class BigQueryMCP:
    """Warehouse backed by Google's managed BigQuery MCP server (execute_sql_readonly only)."""

    dialect = "bigquery"

    def __init__(
        self,
        project: str,
        dataset: str,
        mcp_url: str = BQ_MCP_URL,
        max_bytes: int = 100_000_000,
        token_provider: Callable[[], str] = google_access_token,
    ) -> None:
        self.project = project
        self.dataset = dataset
        self.mcp_url = mcp_url
        self.max_bytes = max_bytes
        self._token_provider = token_provider

    def _headers(self) -> dict[str, str]:
        token = self._token_provider()
        return {
            "Authorization": f"Bearer {token}",
            "x-goog-user-project": self.project,
        }

    def _rest_dry_run_bytes(self, bq_sql: str) -> int:
        url = f"https://bigquery.googleapis.com/bigquery/v2/projects/{self.project}/queries"
        if not url.startswith("https://"):
            raise ValueError("HTTPS required for BigQuery REST dryRun")
        body = json.dumps(
            {"query": bq_sql, "useLegacySql": False, "dryRun": True}
        ).encode("utf-8")
        req = urllib.request.Request(
            url,
            data=body,
            headers={"Content-Type": "application/json", **self._headers()},
            method="POST",
        )
        try:
            with urllib.request.urlopen(req, timeout=30) as resp:  # noqa: S310
                data = json.load(resp)
        except urllib.error.HTTPError as exc:
            detail = (
                exc.read().decode("utf-8", errors="replace") if exc.fp else exc.reason
            )
            raise RuntimeError(
                f"BigQuery dryRun failed ({exc.code}): {detail}"
            ) from exc
        return int(data.get("totalBytesProcessed", 0))

    async def _call_readonly(self, bq_sql: str) -> QueryResult:
        headers = self._headers()
        async with (
            create_mcp_http_client(headers=headers) as http_client,
            streamable_http_client(self.mcp_url, http_client=http_client) as streams,
            ClientSession(streams[0], streams[1]) as session,
        ):
            await session.initialize()
            listed = await session.list_tools()
            tools = {tool.name: tool for tool in getattr(listed, "tools", listed)}
            if ALLOWED_BQ_TOOL not in tools:
                msg = (
                    f"MCP server does not expose {ALLOWED_BQ_TOOL}; "
                    f"available: {sorted(tools)}"
                )
                raise RuntimeError(msg)
            tool_obj = tools[ALLOWED_BQ_TOOL]
            schema_dict = (
                getattr(tool_obj, "input_schema", None)
                or getattr(tool_obj, "inputSchema", None)
                or {}
            )
            props = schema_dict.get("properties") or {}
            sql_key = (
                "query" if "query" in props else ("sql" if "sql" in props else "query")
            )
            project_key = (
                "projectId"
                if "projectId" in props
                else ("project_id" if "project_id" in props else "project")
            )
            dry_key = (
                "dryRun"
                if "dryRun" in props
                else ("dry_run" if "dry_run" in props else None)
            )

            call_args: dict[str, Any] = {sql_key: bq_sql}
            if project_key in props or not props:
                target_key = project_key if project_key in props else "projectId"
                call_args[target_key] = self.project
            if "useLegacySql" in props:
                call_args["useLegacySql"] = False

            estimated = 0
            if self.max_bytes > 0:
                if dry_key:
                    dry_res = await session.call_tool(
                        ALLOWED_BQ_TOOL, {**call_args, dry_key: True}
                    )
                    estimated = int(
                        _extract_mcp_payload(dry_res).get("totalBytesProcessed", 0)
                    )
                else:
                    estimated = self._rest_dry_run_bytes(bq_sql)
                if estimated > self.max_bytes:
                    msg = (
                        f"dryRun estimated {estimated} bytes, "
                        f"above cap {self.max_bytes}"
                    )
                    raise RuntimeError(msg)

            live_res = await session.call_tool(ALLOWED_BQ_TOOL, call_args)
            return _parse_bq_payload(_extract_mcp_payload(live_res), estimated)

    def query(
        self,
        sql: str,
        params: Sequence[Any] | None = None,
        max_rows: int = MAX_RESULT_ROWS,
    ) -> QueryResult:
        """Render GoogleSQL via sqlglot AST binding and run over BigQuery MCP."""
        bq_sql = render_bigquery_sql(sql, params, self.project, self.dataset)
        try:
            asyncio.get_running_loop()
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
                result = pool.submit(asyncio.run, self._call_readonly(bq_sql)).result()
        except RuntimeError:
            result = asyncio.run(self._call_readonly(bq_sql))
        return QueryResult(
            columns=result.columns,
            rows=result.rows[:max_rows],
            bytes_estimated=result.bytes_estimated,
        )


class RefusalError(Exception):
    """Raised when a semantic request is refused by certification, RBAC, or guardrails."""


def _plain_cell(value: Any) -> Any:
    if isinstance(value, decimal.Decimal):
        return round(float(value), 4)
    if isinstance(value, float):
        return round(value, 4)
    if isinstance(value, (dt.date, dt.datetime)):
        return value.isoformat()
    return value


class SemanticService:
    """Governed semantic execution service backing the MCP server tools."""

    def __init__(
        self,
        catalog: Catalog,
        warehouse: DuckDBWarehouse | BigQueryMCP,
        log_path: Path | None = None,
    ) -> None:
        self.catalog = catalog
        self.warehouse = warehouse
        self.compiler = SimpleCompiler(dialect=warehouse.dialect)
        self.log_path = log_path

    def _record_event(
        self, event: str, metric: str = "", role: str = "", note: str = ""
    ) -> None:
        if not self.log_path:
            return
        entry = {"event": event, "metric": metric, "role": role, "note": note}
        with self.log_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(entry) + "\n")

    def _require_certified_metric(self, name: str) -> Metric:
        metric = self.catalog.metrics.get(name)
        if metric is None or metric.state is not State.CERTIFIED or not metric.owner:
            raise RefusalError(f"no certified metric '{name}'")
        return metric

    def list_metrics(self) -> list[dict[str, Any]]:
        """List all certified metrics, dimensions, versions, and linked entities."""
        return [
            {
                "name": metric.name,
                "description": metric.description,
                "dimensions": metric.dimensions,
                "version": metric.version,
                "owner": metric.owner,
                "entity": metric.entity,
            }
            for metric in self.catalog.certified_metrics()
        ]

    def describe_metric(self, name: str) -> dict[str, Any]:
        """Return full lineage, owner, version, and SQL expression for one certified metric."""
        metric = self._require_certified_metric(name)
        return {
            "name": metric.name,
            "description": metric.description,
            "definition": metric.expression,
            "table": metric.table,
            "filters": metric.filters,
            "dimensions": metric.dimensions,
            "entity": metric.entity,
            "owner": metric.owner,
            "version": metric.version,
            "lineage": metric.lineage,
        }

    def query_metric(
        self,
        metric: str,
        as_role: str,
        dims: list[str] | None = None,
        filters: dict[str, Any] | None = None,
        entity_keys: list[int] | None = None,
    ) -> dict[str, Any]:
        """Evaluate RBAC, compile SQL, bind AST literals, and execute read-only."""
        metric_obj = self._require_certified_metric(metric)
        rules = self.catalog.certified_rules()
        decision = evaluate_policy(rules, as_role, metric)
        if not decision.allowed:
            self._record_event("refused", metric, as_role, decision.reason)
            raise RefusalError(f"access denied: {decision.reason}")

        used_dims = set(dims or []) | set((filters or {}).keys())
        blocked = sorted(used_dims & decision.deny_dims)
        if blocked:
            self._record_event("refused", metric, as_role, f"deny_dims:{blocked}")
            raise RefusalError(
                f"dimensions not permitted for role '{as_role}': {blocked}"
            )

        try:
            compiled = self.compiler.compile(
                metric_obj,
                dims=dims,
                filters=filters,
                row_filter=decision.row_filter,
                entity_keys=entity_keys,
            )
            rendered_sql = render_bigquery_sql(
                compiled.sql,
                compiled.params,
                getattr(self.warehouse, "project", "demo-project"),
                getattr(self.warehouse, "dataset", "jaffle_shop"),
            )
            result = self.warehouse.query(compiled.sql, compiled.params)
        except CompileError as exc:
            self._record_event("error", metric, as_role, str(exc))
            raise RefusalError(str(exc)) from exc
        except Exception as exc:
            self._record_event("error", metric, as_role, str(exc))
            raise RefusalError(f"query failed: {exc}") from exc

        self._record_event("query", metric, as_role)
        rows = [[_plain_cell(cell) for cell in row] for row in result.rows]
        active_sql = (
            rendered_sql if self.warehouse.dialect == "bigquery" else compiled.sql
        )
        return {
            "metric": metric_obj.name,
            "version": metric_obj.version,
            "owner": metric_obj.owner,
            "definition": metric_obj.expression,
            "sql": active_sql,
            "bigquery_sql": rendered_sql,
            "columns": result.columns,
            "rows": rows,
            "row_count": len(rows),
            "dry_run_bytes": result.bytes_estimated,
        }

    def _filterable_graph_columns(self) -> dict[str, set[str]]:
        certified_metrics = self.catalog.certified_metrics()
        out: dict[str, set[str]] = defaultdict(set)
        for entity in self.catalog.certified_entities():
            out[entity.name] = {
                dim
                for metric in certified_metrics
                if metric.table == entity.table
                for dim in metric.dimensions
            }
        return out

    def describe_graph(self) -> dict[str, Any]:
        """Return certified entities, filterable columns, and foreign-key relationship edges."""
        cols = self._filterable_graph_columns()
        return {
            "entities": [
                {
                    "name": entity.name,
                    "filterable_columns": sorted(cols[entity.name]),
                }
                for entity in self.catalog.certified_entities()
            ],
            "relationships": [
                {"from": edge.src, "name": edge.rel, "to": edge.dst}
                for edge in self.catalog.certified_edges()
            ],
        }

    def traverse_graph(
        self,
        start: str,
        path: list[str],
        as_role: str,
        filters: dict[str, dict[str, Any]] | None = None,
    ) -> dict[str, Any]:
        """Traverse certified foreign-key edges and return matching entity primary keys."""
        rules = self.catalog.certified_rules()
        if not has_unrestricted_access(rules, as_role):
            raise RefusalError(
                "graph traversal requires a role with unrestricted access"
            )
        allowed_cols = self._filterable_graph_columns()
        denied_dims = {
            dim for rule in rules if rule.role == as_role for dim in rule.deny_dims
        }
        for entity_name, conditions in (filters or {}).items():
            permitted = allowed_cols.get(entity_name, set()) - denied_dims
            disallowed = [col for col in conditions if col not in permitted]
            if disallowed:
                msg = (
                    f"not allowed as a filter on {entity_name}: {disallowed}; "
                    f"allowed: {sorted(permitted)}"
                )
                raise RefusalError(msg)
        graph = EntityGraph(
            self.catalog.certified_entities(), self.catalog.certified_edges()
        )
        try:
            compiled = graph.traverse_sql(start, path, filters)
            result = self.warehouse.query(compiled.sql, compiled.params, max_rows=10000)
        except Exception as exc:
            raise RefusalError(f"graph traversal failed: {exc}") from exc
        keys = [int(row[0]) for row in result.rows]
        self._record_event("query", f"graph:{start}", as_role)
        return {"entity": start, "count": len(keys), "keys": keys}

    def report_gap(
        self, question: str, as_role: str = "", reason: str = ""
    ) -> dict[str, Any]:
        """Record a catalog gap when no certified metric answers the user's question."""
        self._record_event("gap", "", as_role, f"{question} | {reason}")
        return {"recorded": True, "question": question}

    def report_issue(self, metric: str, note: str, as_role: str = "") -> dict[str, Any]:
        """Record a data quality or definition issue reported against a metric."""
        self._record_event("issue", metric, as_role, note)
        return {"recorded": True, "metric": metric}

### Inspect `sqlglot` AST literal binding and `UNNEST` compilation

Before starting the MCP server, let's inspect how `SimpleCompiler` and `render_bigquery_sql` compile entity-scoped metrics (`IN UNNEST([...])`) and neutralize a hostile SQL-injection filter payload (`"card' OR '1'='1"`) by binding it as a `sqlglot` string literal node rather than interpolating strings.

In [ ]:
sys.path.insert(0, str(WORK_DIR.resolve()))
from semlayer_engine import Catalog, SimpleCompiler, render_bigquery_sql

demo_catalog = Catalog.from_yaml(CATALOG_PATH)
revenue_metric = demo_catalog.metrics["revenue"]
bq_compiler = SimpleCompiler(dialect="bigquery")

unnest_compiled = bq_compiler.compile(revenue_metric, entity_keys=[1, 4, 12])
print("Compiled GoogleSQL with foreign-key UNNEST:")
print(
    " ",
    render_bigquery_sql(
        unnest_compiled.sql, unnest_compiled.params, "my-demo-project", BQ_DATASET
    ),
)

HOSTILE_FILTER = "card' OR '1'='1"
hostile_compiled = bq_compiler.compile(
    revenue_metric, filters={"method": HOSTILE_FILTER}
)
print("\nCompiled GoogleSQL with hostile injection payload bound as AST literal:")
print(
    " ",
    render_bigquery_sql(
        hostile_compiled.sql, hostile_compiled.params, "my-demo-project", BQ_DATASET
    ),
)

## 3. Create the local `stdio` MCP server and Gemini agent loop

Now we create `semlayer_workspace/mcp_server.py`, exposing **only** the seven governed semantic tools over `stdio` transport:

- `list_metrics`: Lists `certified` metrics, dimensions, owners, and versions (draft metrics such as `revenue_unfiltered` are excluded).
- `describe_metric`: Inspects the SQL aggregate expression, lineage, owner, and version of one certified metric.
- `query_metric`: Evaluates default-deny `RBAC`, injects `row_filter`, blocks `deny_dims`, compiles parameterized SQL, and executes read-only.
- `describe_graph`: Lists certified entities, filterable columns, and foreign-key relationship names.
- `traverse_graph`: Walks certified foreign-key relationships (such as `Customer -HAS_ORDER-> Order`) and returns matching entity keys.
- `report_gap`: Logs missing metric requests instead of inventing uncertified SQL formulas.
- `report_issue`: Logs suspected metric anomalies for human data owners.

Notice what is **missing**: there is **no raw SQL tool**. Agents cannot bypass `RBAC`, query uncertified tables, or touch `customers.email`.

In [ ]:
%%writefile semlayer_workspace/mcp_server.py
"""Local stdio MCP server exposing only certified semantic tools (no raw SQL tool)."""

from __future__ import annotations

import os
from pathlib import Path
from typing import Any

try:
    from mcp.server.mcpserver import MCPServer as Server
except ImportError:
    from mcp.server.fastmcp import FastMCP as Server

from semlayer_engine import (
    BigQueryMCP,
    Catalog,
    DuckDBWarehouse,
    RefusalError,
    SemanticService,
)

HERE = Path(__file__).resolve().parent
CATALOG = Catalog.from_yaml(HERE / "catalog.yaml")

BACKEND = os.environ.get("SEMLAYER_WAREHOUSE", "duckdb")
MAX_BYTES = int(os.environ.get("SEMLAYER_BQ_MAX_BYTES", "100000000"))

if BACKEND == "bigquery":
    WAREHOUSE: DuckDBWarehouse | BigQueryMCP = BigQueryMCP(
        project=os.environ["SEMLAYER_BQ_PROJECT"],
        dataset=os.environ.get("SEMLAYER_BQ_DATASET", "jaffle_shop"),
        max_bytes=MAX_BYTES,
    )
else:
    WAREHOUSE = DuckDBWarehouse(
        db_path=Path(
            os.environ.get("SEMLAYER_DUCKDB_PATH", str(HERE / "jaffle.duckdb"))
        ),
        max_bytes=MAX_BYTES,
    )

SERVICE = SemanticService(
    catalog=CATALOG,
    warehouse=WAREHOUSE,
    log_path=HERE / "usage.jsonl",
)

mcp = Server("certified-bigquery-mcp")


def _guard(func: Any, *args: Any, **kwargs: Any) -> dict[str, Any]:
    try:
        return func(*args, **kwargs)
    except RefusalError as exc:
        return {"error": str(exc)}


@mcp.tool()
def list_metrics() -> list[dict[str, Any]]:
    """List certified metrics, their allowed dimensions, owners, and versions."""
    return SERVICE.list_metrics()


@mcp.tool()
def describe_metric(name: str) -> dict[str, Any]:
    """Return definition, owner, version, and column lineage of one certified metric."""
    return _guard(SERVICE.describe_metric, name)


@mcp.tool()
def query_metric(
    metric: str,
    as_role: str,
    dims: list[str] | None = None,
    filters: dict[str, str] | None = None,
    entity_keys: list[int] | None = None,
) -> dict[str, Any]:
    """Compute a certified metric for `as_role`. Cite metric name, version, and owner."""
    return _guard(
        SERVICE.query_metric,
        metric=metric,
        as_role=as_role,
        dims=dims,
        filters=filters,
        entity_keys=entity_keys,
    )


@mcp.tool()
def describe_graph() -> dict[str, Any]:
    """List certified entities, filterable columns, and relationship names."""
    return SERVICE.describe_graph()


@mcp.tool()
def traverse_graph(
    start: str,
    path: list[str],
    as_role: str,
    filters: dict[str, dict[str, str]] | None = None,
) -> dict[str, Any]:
    """Traverse foreign-key relationships from `start` along `path` and return entity keys."""
    return _guard(
        SERVICE.traverse_graph,
        start=start,
        path=path,
        as_role=as_role,
        filters=filters,
    )


@mcp.tool()
def report_gap(question: str, as_role: str = "", reason: str = "") -> dict[str, Any]:
    """Call when no certified metric can answer the user's question."""
    return SERVICE.report_gap(question=question, as_role=as_role, reason=reason)


@mcp.tool()
def report_issue(metric: str, note: str, as_role: str = "") -> dict[str, Any]:
    """Report a suspected data quality or metric definition issue."""
    return SERVICE.report_issue(metric=metric, note=note, as_role=as_role)


if __name__ == "__main__":
    mcp.run(transport="stdio")

### Define the Gemini MCP agent loop

Following the `intro_to_mcp.ipynb` pattern, `run_agent_loop` discovers tools dynamically from the active MCP `ClientSession`, declares them to Gemini via `types.FunctionDeclaration`, executes requested MCP tool calls over `stdio`, and feeds structured `FunctionResponse` parts back to Gemini until the model produces its final governed answer.

In [ ]:
SYSTEM_INSTRUCTION = " ".join(
    [
        "You are a governed analytics assistant connected to a certification-gated",
        "semantic layer MCP server. Never invent metric formulas. Always pass the",
        "caller's exact role in `as_role`. When answering with a metric, cite its",
        "certified value, `version`, `owner`, and compiled `sql`. If `query_metric`",
        "returns an access-denied or byte-cap error, report the refusal clearly.",
        "For cross-entity conditions (such as customers with returned orders), call",
        "`describe_graph`, then `traverse_graph`, and pass the returned `keys` as",
        "`entity_keys` to `query_metric`.",
    ]
)


def make_server_params(max_bytes: int = BQ_MAX_BYTES) -> StdioServerParameters:
    """Build StdioServerParameters for the local semantic MCP server."""
    env = {
        **os.environ,
        "SEMLAYER_WAREHOUSE": ACTIVE_BACKEND,
        "SEMLAYER_BQ_PROJECT": PROJECT_ID,
        "SEMLAYER_BQ_DATASET": BQ_DATASET,
        "SEMLAYER_BQ_MAX_BYTES": str(max_bytes),
        "SEMLAYER_DUCKDB_PATH": str(DUCKDB_PATH.resolve()),
    }
    return StdioServerParameters(
        command=sys.executable,
        args=[str((WORK_DIR / "mcp_server.py").resolve())],
        env=env,
    )


async def _execute_tool_calls(
    function_calls: list[types.FunctionCall], session: ClientSession
) -> list[types.Part]:
    """Execute Gemini function calls against the MCP ClientSession."""
    response_parts: list[types.Part] = []
    for func_call in function_calls:
        tool_name = func_call.name or ""
        args = func_call.args if isinstance(func_call.args, dict) else {}
        print(f"  [MCP call] {tool_name}({json.dumps(args)})")
        try:
            tool_result = await session.call_tool(tool_name, args)
            result_text = ""
            if getattr(tool_result, "content", None):
                result_text = getattr(tool_result.content[0], "text", "") or ""
            is_err = getattr(tool_result, "is_error", False) or getattr(
                tool_result, "isError", False
            )
            payload: dict[str, Any] = (
                {"error": result_text or f"Tool '{tool_name}' failed."}
                if is_err
                else {"result": result_text}
            )
            print(f"  [MCP result] {result_text[:180]}")
        except Exception as exc:  # noqa: BLE001
            payload = {"error": f"{type(exc).__name__}: {exc}"}
        response_parts.append(
            types.Part.from_function_response(name=tool_name, response=payload)
        )
    return response_parts


async def run_agent_loop(
    prompt: str,
    genai_client: genai.Client,
    session: ClientSession,
    model_id: str = MODEL_ID,
    max_tool_turns: int = 6,
) -> types.GenerateContentResponse:
    """Run a multi-turn Gemini tool-calling loop over an active MCP ClientSession."""
    await session.initialize()
    listed = await session.list_tools()
    mcp_tools = getattr(listed, "tools", listed)

    gemini_tools = types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name=tool.name,
                description=tool.description,
                parameters_json_schema=(
                    getattr(tool, "input_schema", None)
                    or getattr(tool, "inputSchema", None)
                ),
            )
            for tool in mcp_tools
        ]
    )

    contents: list[types.Content] = [
        types.Content(role="user", parts=[types.Part(text=prompt)])
    ]
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        temperature=0.0,
        tools=[gemini_tools],
    )

    response = await genai_client.aio.models.generate_content(
        model=model_id,
        contents=contents,
        config=config,
    )
    if not response.candidates or not response.candidates[0].content:
        return response

    latest_content = response.candidates[0].content
    contents.append(latest_content)
    pending_calls = [
        part.function_call
        for part in (latest_content.parts or [])
        if part.function_call
    ]

    turn = 0
    while pending_calls and turn < max_tool_turns:
        turn += 1
        tool_parts = await _execute_tool_calls(pending_calls, session)
        contents.append(types.Content(role="user", parts=tool_parts))
        response = await genai_client.aio.models.generate_content(
            model=model_id,
            contents=contents,
            config=config,
        )
        if not response.candidates or not response.candidates[0].content:
            break
        latest_content = response.candidates[0].content
        contents.append(latest_content)
        pending_calls = [
            part.function_call
            for part in (latest_content.parts or [])
            if part.function_call
        ]

    return response


async def ask_governed_agent(prompt: str, max_bytes: int = BQ_MAX_BYTES) -> str:
    """Launch the stdio MCP server and run one governed Gemini prompt."""
    if not PROJECT_ID or PROJECT_ID in {"[your-project-id]", "None"}:
        return (
            "Set `PROJECT_ID` in 'Set Google Cloud project information' above "
            "to execute live Gemini tool-calling turns on Agent Platform."
        )
    server_params = make_server_params(max_bytes=max_bytes)
    async with (
        stdio_client(server_params) as (read_stream, write_stream),
        ClientSession(read_stream, write_stream) as session,
    ):
        final_response = await run_agent_loop(prompt, client, session)
        return final_response.text or ""

## 4. Demonstrate the 5 governed analytics scenarios

We now verify all five governance guarantees over the `stdio` MCP boundary:

1. **Authorized metric (`finance` -> `revenue`)**: Returns the certified value (`26035.71`), `version` (`2`), `owner` (`alice`), and compiled SQL while hiding uncertified draft metrics (`revenue_unfiltered`).
2. **Default-deny refusal (`eu_analyst` -> `revenue`)**: Refuses immediately (`access denied: no allow rule for role 'eu_analyst'`).
3. **Row-level filtering (`eu_analyst` -> `customer_count`)**: Automatically injects `WHERE (region = 'EU')` from the certified `row_filter` rule, returning `24` EU customers instead of all `60` customers.
4. **Multi-hop foreign-key entity-graph traversal (`Customer -HAS_ORDER-> Order` returned orders -> `revenue`)**: Traverses `HAS_ORDER` filtered to `status = 'returned'` (`26` customer keys) and computes `revenue` scoped via `IN UNNEST(...)` (`12384.46`).
5. **Hostile SQL-injection neutralization + `dryRun` byte-cap check**: Binds `"card' OR '1'='1"` as a safe `sqlglot` string literal (`[[None]]` with zero data leakage) and blocks queries whose `dryRun` byte estimate exceeds `max_bytes` before execution.

### Scenario A: Authorized metric (`finance` -> `revenue`)

Role `finance` holds a certified `allow` rule (`metric: "*"`), so `query_metric` compiles `SUM(amount)` filtered to `status = 'success'` and returns provenance metadata (`version`, `owner`, and `sql`).

In [ ]:
async def verify_mcp_tool(
    tool_name: str, tool_args: dict[str, Any], max_bytes: int = BQ_MAX_BYTES
) -> Any:
    """Call a single tool over stdio MCP and parse its JSON response."""
    async with (
        stdio_client(make_server_params(max_bytes=max_bytes)) as (
            read_stream,
            write_stream,
        ),
        ClientSession(read_stream, write_stream) as session,
    ):
        await session.initialize()
        if tool_name == "__list_tools__":
            listed = await session.list_tools()
            return [t.name for t in getattr(listed, "tools", listed)]
        res = await session.call_tool(tool_name, tool_args)
        texts = [
            getattr(c, "text", "")
            for c in getattr(res, "content", [])
            if getattr(c, "text", "")
        ]
        return (
            json.loads("\n".join(texts))
            if len(texts) == 1
            else [json.loads(t) for t in texts]
        )


exposed_tools = await verify_mcp_tool("__list_tools__", {})
print("Exposed MCP tools (no raw SQL tool):", exposed_tools)

scenario_a_mcp = await verify_mcp_tool(
    "query_metric", {"metric": "revenue", "as_role": "finance"}
)
print("MCP response:", json.dumps(scenario_a_mcp, indent=2))

scenario_a_agent = await ask_governed_agent(
    "As role 'finance', what is our total certified revenue? "
    "Include the metric version, owner, and compiled SQL."
)
display(Markdown(scenario_a_agent))

### Scenario B: Default-deny refusal (`eu_analyst` -> `revenue`)

Role `eu_analyst` only holds certified `allow` rules for `customer_count` and `order_count`. Requesting `revenue` triggers a deterministic default-deny refusal.

In [ ]:
scenario_b_mcp = await verify_mcp_tool(
    "query_metric", {"metric": "revenue", "as_role": "eu_analyst"}
)
print("MCP refusal:", json.dumps(scenario_b_mcp, indent=2))

scenario_b_agent = await ask_governed_agent(
    "As role 'eu_analyst', what is our total revenue?"
)
display(Markdown(scenario_b_agent))

### Scenario C: Row-level filtering (`eu_analyst` -> `customer_count`)

When `eu_analyst` queries `customer_count`, `evaluate_policy` attaches `row_filter: "region = 'EU'"`, returning `24` EU customers rather than the unfiltered `60` total customers.

In [ ]:
scenario_c_mcp = await verify_mcp_tool(
    "query_metric", {"metric": "customer_count", "as_role": "eu_analyst"}
)
print("MCP row-filtered response:", json.dumps(scenario_c_mcp, indent=2))

scenario_c_agent = await ask_governed_agent(
    "As role 'eu_analyst', how many customers do we have?"
)
display(Markdown(scenario_c_agent))

### Scenario D: Multi-hop foreign-key entity-graph traversal (`Customer -HAS_ORDER-> Order` returned orders -> `revenue`)

A common raw text-to-SQL failure mode is joining `payments.order_id = orders.id` when the business question asks for **total revenue from customers who have ever returned an order** (`Customer` entity semantics). Here, `traverse_graph` walks `Customer -HAS_ORDER-> Order` with `filters={"Order": {"status": "returned"}}` to collect the `26` matching customer keys, and `query_metric` scopes `revenue` via `customer_id IN UNNEST(...)`.

In [ ]:
graph_hop = await verify_mcp_tool(
    "traverse_graph",
    {
        "start": "Customer",
        "path": ["HAS_ORDER"],
        "as_role": "finance",
        "filters": {"Order": {"status": "returned"}},
    },
)
print(f"Matched Customer keys ({graph_hop['count']}): {graph_hop['keys'][:10]}...")

scenario_d_mcp = await verify_mcp_tool(
    "query_metric",
    {
        "metric": "revenue",
        "as_role": "finance",
        "entity_keys": graph_hop["keys"],
    },
)
print("MCP entity-scoped revenue:", json.dumps(scenario_d_mcp, indent=2))

scenario_d_agent = await ask_governed_agent(
    "As role 'finance', use the entity graph to find all Customers who have an "
    "Order with status 'returned', then compute certified 'revenue' for those customers."
)
display(Markdown(scenario_d_agent))

### Scenario E: Hostile SQL-injection filter payload (`"card' OR '1'='1"`) + `dryRun` byte-cap check

Finally, we test two defense-in-depth warehouse guardrails:

1. **Hostile filter injection (`"card' OR '1'='1"`)**: Instead of tautology-bypassing `WHERE status = 'success'`, `sqlglot` binds the string as a literal AST node (`method = 'card\' OR \'1\'=\'1'`), matching zero payments (`[[None]]`) and leaking zero rows.
2. **Pre-execution `dryRun` byte cap**: Setting `max_bytes = 1` causes `BigQueryMCP` (and `DuckDBWarehouse` in offline mode) to inspect `totalBytesProcessed` via `dryRun` first and abort the query before scanning data.

In [ ]:
scenario_e_injection = await verify_mcp_tool(
    "query_metric",
    {
        "metric": "revenue",
        "as_role": "finance",
        "filters": {"method": HOSTILE_FILTER},
    },
)
print(
    "Hostile payload safely bound (rows=[[None]], zero leakage):",
    json.dumps(scenario_e_injection, indent=2),
)

scenario_e_byte_cap = await verify_mcp_tool(
    "query_metric",
    {"metric": "revenue", "as_role": "finance"},
    max_bytes=1,
)
print("Pre-flight dryRun byte-cap refusal:", json.dumps(scenario_e_byte_cap, indent=2))

scenario_e_agent = await ask_governed_agent(
    f"As role 'finance', query 'revenue' filtered to payment method {HOSTILE_FILTER!r} "
    "and report what happens."
)
display(Markdown(scenario_e_agent))

## 5. Governance benchmark: certified semantic MCP vs. raw text-to-SQL

Empirical evaluation across a **33-question tuned benchmark** and an **18-question held-out benchmark** (covering single-metric, grouped, filtered, `RBAC` role-restricted, uncertified-concept abstention, and multi-hop entity-graph questions on `jaffle_shop`) highlights why stronger models alone do not replace a semantic governance layer:

| Evaluation Set | Warehouse | Arm | Correct Answers | Real Refusals | Wrong Answers | Overall Score |
| --- | --- | --- | --- | --- | --- | --- |
| Tuned (`33`) | BigQuery MCP | Certified `semlayer` (Gemini) | `25/25` | `8/8` | **`0`** | **`33/33` (`100%`)** |
| Tuned (`33`) | `DuckDB` | Certified `semlayer` (Gemini) | `25/25` | `8/8` | **`0`** | **`33/33` (`100%`)** |
| Tuned (`33`) | `DuckDB` | Raw SQL + column hints (Gemini) | `22/25` | **`0/8`** | `3` | `22/33` (`67%`) |
| Tuned (`33`) | `DuckDB` | Raw SQL schema-only (Gemini) | `15/25` | **`0/8`** | `9` (+`1` SQL err) | `15/33` (`45%`) |
| Held-out (`18`) | BigQuery MCP | Certified `semlayer` (Gemini) | `12/13` | `5/5` | **`0`** | **`17/18` (`94%`)** |
| Held-out (`18`) | `DuckDB` | Certified `semlayer` (Gemini) | `13/13` | `4/5` | **`0`** | **`17/18` (`94%`)** |
| Held-out (`18`) | `DuckDB` | Raw SQL + column hints (Gemini) | `11/13` | **`0/5`** | `2` | `11/18` (`61%`) |
| Held-out (`18`) | `DuckDB` | Raw SQL schema-only (Gemini) | `6/13` | **`0/5`** | `7` | `6/18` (`33%`) |

**Key findings:**

- **Zero silent wrong answers on the certified layer**: Across both `DuckDB` and Google's managed BigQuery MCP (`execute_sql_readonly` + `dryRun`), the certification-gated semantic layer produced **`0` wrong answers** on both tuned (`33/33`) and held-out (`17/18`) question sets.
- **Raw text-to-SQL achieved `0/8` and `0/5` real refusals**: Without a semantic MCP boundary, raw text-to-SQL leaked `customers.email`, answered unauthorized `eu_analyst` and `guest` queries, fabricated SQL formulas for uncertified metrics (`churn`, `gross_margin`, `NPS`), and joined `payments.order_id = orders.id` instead of customer-level entity semantics.

## Next steps / Reference implementation

- **Full reference implementation & evaluation harness**: https://github.com/RiturajSingh2004/arth-semlayer (includes query-log conflict detection, schema-drift monitoring, policy replay, `dbt` manifest ingestion, and the full 33-question / 18-question benchmark suite).
- **Introductory MCP pattern**: Review [`gemini/mcp/intro_to_mcp.ipynb`](https://github.com/RiturajSingh2004/arth-semlayer/blob/main/notebooks/intro_to_mcp.ipynb) for additional custom and pre-built MCP server examples.
- **BigQuery MCP documentation**: Learn more about [Google Cloud's managed BigQuery MCP server](https://cloud.google.com/bigquery/docs/use-bigquery-mcp) and IAM least-privilege service account configuration.

## Cleaning up

To clean up all Google Cloud resources used in this project, you can [delete the Google Cloud project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects) you used for the tutorial.

Otherwise, you can delete the demo BigQuery dataset (`jaffle_shop`) and local workspace files created in this tutorial by setting `DELETE_RESOURCES = True` and running the cell below:

- BigQuery dataset: `{PROJECT_ID}.{BQ_DATASET}`
- Local workspace directory: `semlayer_workspace/`

In [ ]:
DELETE_RESOURCES = False  # @param {type: "boolean"}

if DELETE_RESOURCES:
    if (
        ACTIVE_BACKEND == "bigquery"
        and PROJECT_ID
        and PROJECT_ID not in {"[your-project-id]", "None"}
    ):
        try:
            from google.cloud import bigquery

            bq_cleanup_client = bigquery.Client(
                project=PROJECT_ID, location=BQ_LOCATION
            )
            bq_cleanup_client.delete_dataset(
                f"{PROJECT_ID}.{BQ_DATASET}",
                delete_contents=True,
                not_found_ok=True,
            )
            print(f"Deleted BigQuery dataset: {PROJECT_ID}.{BQ_DATASET}")
        except ImportError:
            bq_bin = shutil.which("bq") or shutil.which("bq.cmd")
            if bq_bin:
                subprocess.run(  # noqa: S603
                    [
                        bq_bin,
                        f"--project_id={PROJECT_ID}",
                        "rm",
                        "-r",
                        "-f",
                        "-d",
                        f"{PROJECT_ID}:{BQ_DATASET}",
                    ],
                    check=True,
                )
                print(
                    f"Deleted BigQuery dataset via bq CLI: {PROJECT_ID}:{BQ_DATASET}"
                )

    shutil.rmtree(WORK_DIR, ignore_errors=True)
    print(f"Removed local directory: {WORK_DIR}")
else:
    print("Skipping cleanup. Set DELETE_RESOURCES = True to delete demo resources.")